# Lab 07-05 · Sinusoidal Steady State, Phasors and Impedance

Companion to chapter [07-05](../../chapters/07-05-sinusoidal-steady-state-and-impedance.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import cmath
import math
import shutil
import subprocess
from pathlib import Path
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)
deg = lambda z: math.degrees(cmath.phase(z))


def ngspice_values(*paths):
    vals = {}
    for path in paths:
        out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
        for line in out.splitlines():
            parts = line.split("=")
            if len(parts) >= 2:
                try:
                    vals[parts[0].strip()] = float(parts[1].split()[0])
                except (ValueError, IndexError):
                    pass
    return vals


NG = ngspice_values(*sorted(str(p) for p in Path(".").glob("*.cir"))) if HAVE_NGSPICE else {}

ngspice available: True


## Part 1 — Into steady state

Example 07-05.2 in the time domain: states $v_1$ (capacitor) and $i_L$; node 2 is $v_2 = R_2(i_L + i_s)$.

**🔮 Predict:** how many 1 kHz periods until the node voltages match the phasors to 0.1%? Hint: look at the slowest natural frequency.

In [2]:
R1, C1, L1, R2 = 1e3, 100e-9, 100e-3, 2e3
w = 2 * np.pi * 1e3
A = np.array([[-1 / (R1 * C1), -1 / C1], [1 / L1, -R2 / L1]])
print("natural frequencies (1/s):", np.round(np.linalg.eigvals(A), 1))
slow = min(-np.linalg.eigvals(A).real)
print(f"slowest decay rate {slow:.0f} 1/s: 0.1% after about {np.log(1000) / slow * 1e3:.1f} ms = {np.log(1000) / slow * 1e3:.1f} periods")


def f(t, x):
    vs, i_s = 10 * np.cos(w * t), 0.01 * np.cos(w * t - np.pi / 2)
    return A @ x + np.array([vs / (R1 * C1), -R2 * i_s / L1])


sol = solve_ivp(f, (0, 40e-3), [0, 0], method="DOP853", rtol=1e-10, atol=1e-12, dense_output=True, max_step=5e-6)
t = np.linspace(0, 40e-3, 40001)
x = sol.sol(t)
v2 = R2 * (x[1] + 0.01 * np.cos(w * t - np.pi / 2))
tm = np.linspace(30e-3, 40e-3, 100000, endpoint=False)
xm = sol.sol(tm)
V1 = 2 * np.mean(xm[0] * np.exp(-1j * w * tm))
V2 = 2 * np.mean(R2 * (xm[1] + 0.01 * np.cos(w * tm - np.pi / 2)) * np.exp(-1j * w * tm))
print(f"lock-in: V1 = {abs(V1):.3f} V at {deg(V1):.2f} deg, V2 = {abs(V2):.3f} V at {deg(V2):.2f} deg")
assert abs(abs(V1) - 7.542) < 2e-3 and abs(abs(V2) - 10.80) < 5e-3
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(t * 1e3, v2, lw=0.8, label="$v_2(t)$ simulated")
ax.plot(t * 1e3, abs(V2) * np.cos(w * t + cmath.phase(V2)), "--", lw=0.8, label="phasor steady state")
ax.set_xlabel("time (ms)"); ax.set_ylabel("V"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.show()
if NG:
    print("ngspice AC:", NG["ex2m2"], "V at", NG["ex2p2"], "deg")

natural frequencies (1/s): [-15000.+8660.3j -15000.-8660.3j]
slowest decay rate 15000 1/s: 0.1% after about 0.5 ms = 0.5 periods


lock-in: V1 = 7.542 V at -70.33 deg, V2 = 10.804 V at -56.28 deg
ngspice AC: 10.80448 V at -56.2761 deg


## Part 2 — Two readings of one capacitor

Example 07-05.1's capacitor: 10 µF, 0.1 Ω ESR, 2 nH ESL.

**🔮 Predict:** at what frequency do the series and parallel capacitances differ by 10%?

In [3]:
Cc, Rc, Lc = 10e-6, 0.1, 2e-9
f = np.logspace(2, 6.2, 600)
wv = 2 * np.pi * f
Z = Rc + 1j * wv * Lc + 1 / (1j * wv * Cc)
Cs = np.where(Z.imag < 0, -1 / (wv * Z.imag), np.nan)
Cp = (1 / Z).imag / wv
D = Z.real / np.abs(Z.imag)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogx(f, Cs * 1e6, label="$C_s$ (series mode)")
ax.semilogx(f, Cp * 1e6, label="$C_p$ (parallel mode)")
ax.set_xlabel("test frequency (Hz)"); ax.set_ylabel("reported capacitance (µF)"); ax.set_ylim(0, 20)
ax.legend(fontsize=8); ax.grid(True, which="both", alpha=0.3)
plt.show()
ratio = lambda fr: (lambda z: (-1 / (2 * np.pi * fr * z.imag)) / ((1 / z).imag / (2 * np.pi * fr)))(Rc + 1j * 2 * np.pi * fr * Lc + 1 / (1j * 2 * np.pi * fr * Cc))
f10 = brentq(lambda fr: ratio(fr) - 1.1, 1e3, 1e6)
print(f"Cs/Cp = 1.1 at {f10 / 1e3:.1f} kHz")
assert 40e3 < f10 < 60e3

Cs/Cp = 1.1 at 50.2 kHz


At low frequency $D$ is tiny and the two modes agree; as the ESR becomes comparable with the reactance they diverge, and near self-resonance (1.125 MHz) both lose meaning. Datasheets therefore state the test frequency, and a part's "capacitance" at 100 kHz is not its 120 Hz value.

## Part 3 — The phasor diagram

Example 07-05.4, and the same current magnitude with a capacitive (leading) load.

**🔮 Predict:** with a leading current, is the source voltage above or below 23 V?

In [4]:
VL = 23.0
Zline = 0.5 + 0.3j
for name, I in (("inductive (lagging)", 1.0 * cmath.exp(-1j * math.radians(36.87))),
                ("capacitive (leading)", 1.0 * cmath.exp(1j * math.radians(36.87)))):
    VS = VL + I * Zline
    print(f"{name:21s}: |VS| = {abs(VS):.3f} V, angle {deg(VS):.3f} deg")
VS_lead = VL + cmath.exp(1j * math.radians(36.87)) * Zline
assert abs(abs(VS_lead) - 23.23) < 0.005

inductive (lagging)  : |VS| = 23.580 V, angle -0.146 deg
capacitive (leading) : |VS| = 23.226 V, angle 1.332 deg


With a leading current the reactive drop $jX\mathbf{I}$ turns partly against the load voltage: the source needs to supply only 23.23 V instead of 23.58 V. On a long cable with a large capacitive current the far-end voltage can even exceed the source's.

## Part 4 — Phase changes shape

A square wave (odd harmonics up to the 15th) through the all-pass of Example 07-05.6.

**🔮 Predict:** does the RMS value change? The peak?

In [5]:
R, C = 10e3, 1 / (2 * np.pi * 1e3 * 10e3)
t = np.linspace(0, 2e-3, 20000, endpoint=False)
x = np.zeros_like(t); y = np.zeros_like(t)
for k in range(1, 16, 2):
    wk = 2 * np.pi * 1e3 * k
    H = (1 - 1j * wk * R * C) / (1 + 1j * wk * R * C)
    x += (4 / (np.pi * k)) * np.cos(wk * t)
    y += (4 / (np.pi * k)) * abs(H) * np.cos(wk * t + cmath.phase(H))
print(f"RMS in {np.sqrt(np.mean(x**2)):.4f} V, out {np.sqrt(np.mean(y**2)):.4f} V; peak in {x.max():.3f} V, out {y.max():.3f} V")
assert abs(np.sqrt(np.mean(x**2)) - np.sqrt(np.mean(y**2))) < 1e-9
fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.plot(t * 1e3, x, lw=0.8, label="input (8 harmonics)")
ax.plot(t * 1e3, y, lw=0.8, label="all-pass output")
ax.set_xlabel("time (ms)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.show()
if NG:
    print("ngspice: all-pass phase at 3 kHz", NG["ex6p"], "deg")

RMS in 0.9873 V, out 0.9873 V; peak in 2.574 V, out 1.309 V
ngspice: all-pass phase at 3 kHz -143.13 deg


## Guided exercise

Raise the ESL to 20 nH and predict the series-mode capacitance at 100 kHz and the new self-resonant frequency before running.

In [6]:
w = 2 * np.pi * 100e3
Z = 0.1 + 1j * w * 20e-9 + 1 / (1j * w * 10e-6)
print(f"Cs at 100 kHz = {-1 / (w * Z.imag) * 1e6:.2f} uF; self-resonance {1 / (2 * np.pi * np.sqrt(20e-9 * 10e-6)) / 1e3:.1f} kHz")

Cs at 100 kHz = 10.86 uF; self-resonance 355.9 kHz


## Your turn

1. What is the admittance of Example 07-05.1's capacitor at 1 kHz, as $G$ and $B$?
2. In Part 3, what load phase angle (with 1 A) makes the source voltage exactly equal to 23 V?
3. Design an all-pass with a −90° point at 10 kHz using a 1 nF capacitor. What resistor do you need?

<details><summary>Answers</summary>

1. $Y = 1/Z = 0.3948 + j62.83$ mS: $G$ = **0.3948 mS**, $B$ = **62.83 mS** (essentially $\omega C$).
2. With a leading current the in-phase part of the drop, $I(R\cos\varphi - X\sin\varphi)$, turns negative and can cancel the small rise that the quadrature part causes; solving $\lvert\mathbf{V}_S\rvert = 23$ V numerically gives a lead of **59.8°**.
3. $R = 1/(2\pi\times10^4\times10^{-9}) = $ **15.92 kΩ**.

</details>

In [7]:
Z1 = 0.1 + 1j * 2 * np.pi * 1e3 * 2e-9 + 1 / (1j * 2 * np.pi * 1e3 * 10e-6)
Y1 = 1 / Z1
print(f"1) G = {Y1.real * 1e3:.4f} mS, B = {Y1.imag * 1e3:.2f} mS")
assert abs(Y1.real * 1e3 - 0.3948) < 1e-4 and abs(Y1.imag * 1e3 - 62.83) < 0.01
phi = brentq(lambda a: abs(23 + cmath.exp(1j * math.radians(a)) * (0.5 + 0.3j)) - 23, 1, 89)
print(f"2) lead angle for |VS| = 23 V: {phi:.1f} deg")
assert abs(phi - 59.76) < 0.01
print(f"3) R = {1 / (2 * np.pi * 1e4 * 1e-9) / 1e3:.2f} kohm")

1) G = 0.3948 mS, B = 62.83 mS
2) lead angle for |VS| = 23 V: 59.8 deg
3) R = 15.92 kohm
